# Getrennte RBF-SVM: Preprocessing-Screen

**Hypothese:** Die Preprocessing-Konfiguration der hierarchischen SVM oder eine Formtransformation verbessert die kombinierte CV-BAcc der unabhängigen Crop- und Stage-SVMs gegenüber der bisherigen Konfiguration.

Der Validierungssplit wurde bereits für `independent_svm` verwendet und bleibt in diesem Notebook vollständig unberührt. Alle Varianten werden ausschließlich auf denselben fünf Trainingsfolds verglichen.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.experiment import TuneConfig, tune
from awp2.models import make_independent_svm
from awp2.preprocessing import PreprocessingConfig

REFERENCE_CV_BACC_COMBINED = 0.8152
CROP_C = 300.0
CROP_GAMMA = 0.003
STAGE_C = 30.0
STAGE_GAMMA = 0.01

PREPROCESSING_VARIANTS = {
    "legacy_scale_cyclical": PreprocessingConfig(
        scale=True,
        use_meta=True,
        use_cyclic_month=True,
    ),
    "numeric_month": PreprocessingConfig(
        scale=True,
        use_meta=True,
        use_cyclic_month=False,
    ),
    "hierarchical_reference": PreprocessingConfig(
        use_spectral_standard_scale=True,
        use_meta=True,
        use_cyclic_month=True,
    ),
    "savgol": PreprocessingConfig(
        use_spectral_standard_scale=True,
        use_meta=True,
        use_cyclic_month=True,
        use_savgol=True,
        savgol_window_length=5,
        savgol_polyorder=2,
    ),
    "snv_mean_albedo": PreprocessingConfig(
        use_spectral_standard_scale=True,
        use_meta=True,
        use_cyclic_month=True,
        use_spectral_snv=True,
        use_mean_albedo=True,
    ),
}

## CV-only Vergleich

Die Hyperparameter bleiben auf dem bisherigen CV-Gewinner fest, damit der Screen ausschließlich die Vorverarbeitung vergleicht. `legacy_scale_cyclical` reproduziert den bereits bewerteten Lauf exakt. `numeric_month` ist die Monatskodierungs-Ablation. `hierarchical_reference` trennt die Modellierungsfrage von der abweichenden Vorverarbeitung. Savitzky-Golay prüft lokale spektrale Glättung; SNV plus Mean Albedo prüft die Kurvenform und behält Helligkeit als separates Merkmal.

Ein Scoregewinn von weniger als 0,005 gegenüber der Referenz wird als Rauschen behandelt. Dieses Notebook enthält absichtlich keinen `run()`-Aufruf.

In [ ]:
PARAM_GRID = {
    "crop_estimator__C": [CROP_C],
    "crop_estimator__gamma": [CROP_GAMMA],
    "stage_estimator__C": [STAGE_C],
    "stage_estimator__gamma": [STAGE_GAMMA],
}

screen_results = []
for variant_name, preprocessing in PREPROCESSING_VARIANTS.items():
    tuned = tune(
        make_independent_svm(),
        TuneConfig(
            name=f"independent_svm_prep_{variant_name}_search",
            approach="independent",
            description=(
                "CV-only Preprocessing-Screen der unabhängigen Crop- und Stage-RBF-SVM; "
                f"Variante: {variant_name}."
            ),
            preprocessing=preprocessing,
            param_grid=PARAM_GRID,
        ),
    )
    screen_results.append(
        {
            "variant": variant_name,
            "cv_bacc_combined": tuned.best_score,
            "cv_std": tuned.candidates.loc[0, "std"],
            "delta_to_reference": tuned.best_score - REFERENCE_CV_BACC_COMBINED,
            "tuning_run": tuned.run_id,
        }
    )

screen = pd.DataFrame(screen_results).sort_values("cv_bacc_combined", ascending=False)
display(screen)

## Entscheidung

Dokumentiert werden die vier CV-only Läufe in der zweiten Tabelle von `docs/modelle/experimente.md`. Der Holdout bleibt geschlossen. Nur wenn eine Variante die Referenz um mindestens 0,005 übertrifft, folgt eine separate, kleine CV-Suche von `C` und `gamma` um diese Konfiguration. Ein weiterer `run()`-Aufruf für den unabhängigen Ansatz ist nicht vorgesehen.

In [ ]:
best = screen.iloc[0]
decision = {
    "best_variant": best["variant"],
    "best_cv_bacc_combined": best["cv_bacc_combined"],
    "delta_to_reference": best["delta_to_reference"],
    "clears_screen_threshold": best["delta_to_reference"] >= 0.005,
    "holdout_used": False,
}
decision